# Account Usage Tape Dashboard

This notebook reads the `agentos` account-usage history tape at `~/.agentos/chat-account-stats/history` and turns it into compact rows for visualization.

It is intentionally dependency-light:
- no `pandas`
- no `matplotlib`
- uses `IPython`, `ipywidgets`, HTML, and inline SVG
- normalizes older provider-native snapshots and newer `usage_over_time` snapshots into one view

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import html
import json
import math
import sys
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any

from IPython.display import HTML, Markdown, clear_output, display
import ipywidgets as widgets

PALETTE = [
    "#0f766e",
    "#2563eb",
    "#ea580c",
    "#7c3aed",
    "#dc2626",
    "#0891b2",
    "#65a30d",
    "#c2410c",
]
PROVIDER_COLORS = {
    "codex": "#2563eb",
    "claude-code": "#ea580c",
    "copilot": "#0f766e",
    "copilot-cli": "#7c3aed",
}
SURFACE = "#fbf7ef"
PANEL = "#ffffff"
GRID = "#ddd6c8"
TEXT = "#231f1a"
MUTED = "#6e665e"

display(
    HTML(
        """
        <style>
          .usage-wrap {
            font-family: "Avenir Next", "Helvetica Neue", "Trebuchet MS", sans-serif;
            color: #231f1a;
          }
          .usage-grid {
            display: grid;
            gap: 14px;
          }
          .usage-cards {
            grid-template-columns: repeat(auto-fit, minmax(180px, 1fr));
          }
          .usage-card {
            background: #ffffff;
            border: 1px solid #e7dece;
            border-radius: 18px;
            padding: 14px 16px;
            box-shadow: 0 10px 30px rgba(35, 31, 26, 0.06);
          }
          .usage-kicker {
            font-size: 11px;
            letter-spacing: 0.08em;
            text-transform: uppercase;
            color: #6e665e;
            margin-bottom: 6px;
          }
          .usage-value {
            font-size: 28px;
            line-height: 1.1;
            font-weight: 700;
          }
          .usage-note {
            font-size: 12px;
            color: #6e665e;
            margin-top: 6px;
          }
          .usage-panel {
            background: #ffffff;
            border: 1px solid #e7dece;
            border-radius: 22px;
            padding: 14px 16px 18px 16px;
            box-shadow: 0 10px 30px rgba(35, 31, 26, 0.06);
            margin: 10px 0 16px 0;
          }
          .usage-panel h3 {
            margin: 0 0 2px 0;
            font-size: 18px;
          }
          .usage-panel p {
            margin: 0 0 12px 0;
            color: #6e665e;
            font-size: 13px;
          }
          .usage-legend {
            display: flex;
            flex-wrap: wrap;
            gap: 10px 14px;
            margin-top: 10px;
            font-size: 12px;
            color: #403a33;
          }
          .usage-legend-item {
            display: inline-flex;
            align-items: center;
            gap: 6px;
          }
          .usage-dot {
            width: 10px;
            height: 10px;
            border-radius: 999px;
            display: inline-block;
          }
          .usage-table {
            width: 100%;
            border-collapse: collapse;
            font-size: 12px;
          }
          .usage-table th,
          .usage-table td {
            text-align: left;
            padding: 8px 10px;
            border-bottom: 1px solid #eee6d8;
            vertical-align: top;
          }
          .usage-table th {
            color: #6e665e;
            font-size: 11px;
            letter-spacing: 0.06em;
            text-transform: uppercase;
          }
        </style>
        """
    )
)

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "agentos").exists():
            return candidate
    return current

REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

def resolve_tape_root() -> Path:
    candidates = [
        Path.home() / ".agentos" / "chat-account-stats" / "history",
        REPO_ROOT / ".agentos" / "chat-account-stats" / "history",
    ]
    for candidate in candidates:
        if (candidate / "segments").exists():
            return candidate
    return candidates[0]

TAPE_ROOT = resolve_tape_root()
SEGMENTS_DIR = TAPE_ROOT / "segments"

display(Markdown(f"Using tape root: `{TAPE_ROOT}`"))

In [ ]:
def maybe_str(value: object) -> str | None:
    if value is None:
        return None
    if not isinstance(value, str):
        value = str(value)
    value = value.strip()
    return value or None

def maybe_int(value: object) -> int | None:
    if value is None or value == "":
        return None
    if isinstance(value, bool):
        return int(value)
    if isinstance(value, int):
        return value
    if isinstance(value, float):
        return int(round(value))
    try:
        return int(round(float(str(value))))
    except (TypeError, ValueError):
        return None

def parse_utc(value: object) -> dt.datetime | None:
    text = maybe_str(value)
    if text is None:
        return None
    try:
        parsed = dt.datetime.fromisoformat(text.replace("Z", "+00:00"))
    except ValueError:
        return None
    if parsed.tzinfo is None:
        parsed = parsed.replace(tzinfo=dt.timezone.utc)
    return parsed.astimezone(dt.timezone.utc)

def iso_utc(value: dt.datetime | None) -> str | None:
    if value is None:
        return None
    return value.astimezone(dt.timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")

def pretty_ts(value: dt.datetime | None) -> str:
    if value is None:
        return "n/a"
    return value.astimezone().strftime("%Y-%m-%d %H:%M")

def provider_title(provider: str | None) -> str:
    mapping = {
        "codex": "Codex",
        "claude-code": "Claude Code",
        "copilot": "Copilot",
        "copilot-cli": "Copilot CLI",
    }
    return mapping.get(provider or "", provider or "unknown")

def codex_window_name(window_duration_mins: int | None, window_key: str | None) -> str:
    if window_duration_mins == 300:
        return "burst"
    if window_duration_mins == 1440:
        return "daily"
    if window_duration_mins == 10080:
        return "weekly"
    if window_duration_mins is not None:
        return f"{window_duration_mins}m"
    return window_key or "usage"

def codex_window_label(window_duration_mins: int | None, window_key: str | None) -> str:
    if window_duration_mins == 300:
        return "Burst"
    if window_duration_mins == 1440:
        return "24h"
    if window_duration_mins == 10080:
        return "Weekly"
    if window_duration_mins is not None:
        return f"{window_duration_mins}m"
    return window_key or "usage"

def claude_limit_name(limit_type: str | None) -> str:
    mapping = {
        "five_hour": "burst",
        "seven_day": "weekly",
        "seven_day_opus": "weekly_opus",
        "seven_day_sonnet": "weekly_sonnet",
        "overage": "extra",
    }
    return mapping.get(limit_type or "", limit_type or "usage")

def claude_limit_label(limit_type: str | None) -> str:
    mapping = {
        "five_hour": "Burst",
        "seven_day": "Weekly",
        "seven_day_opus": "Weekly Opus",
        "seven_day_sonnet": "Weekly Sonnet",
        "overage": "extra",
    }
    return mapping.get(limit_type or "", limit_type or "usage")

def copilot_limit_name(limit_label: str | None) -> str:
    label = (limit_label or "usage").strip().lower()
    if "premium" in label:
        return "premium"
    if "chat" in label:
        return "chat"
    return label.replace(" ", "_")

def compute_elapsed_percent(
    recorded_at: dt.datetime,
    period_started_at: dt.datetime | None,
    period_ends_at: dt.datetime | None,
) -> int | None:
    if period_started_at is None or period_ends_at is None:
        return None
    if period_ends_at <= period_started_at:
        return None
    span = (period_ends_at - period_started_at).total_seconds()
    if span <= 0:
        return None
    elapsed = (recorded_at - period_started_at).total_seconds()
    ratio = max(0.0, min(1.0, elapsed / span))
    return round(ratio * 100)

def normalize_limit_payload(limit_payload: dict[str, Any], provider: str) -> dict[str, Any] | None:
    usage_over_time = limit_payload.get("usage_over_time")
    if isinstance(usage_over_time, dict):
        limit_name = maybe_str(limit_payload.get("limit_name"))
        limit_label = maybe_str(limit_payload.get("limit_label"))
        provider_limit = limit_payload.get("provider_limit")
        if not isinstance(provider_limit, dict):
            provider_limit = {}
        if limit_name is None and provider == "copilot":
            limit_name = copilot_limit_name(limit_label)
        if limit_name is None:
            limit_name = "usage"
        if limit_label is None:
            limit_label = limit_name.replace("_", " ").title()
        return {
            "limit_class": maybe_str(limit_payload.get("limit_class")) or "usage_over_time",
            "limit_name": limit_name,
            "limit_label": limit_label,
            "used_percent": maybe_int(usage_over_time.get("used_percent")),
            "period_started_at": parse_utc(usage_over_time.get("period_started_at")),
            "period_ends_at": parse_utc(usage_over_time.get("period_ends_at") or usage_over_time.get("resets_at")),
            "period_duration_mins": maybe_int(usage_over_time.get("period_duration_mins")),
            "resets_at": parse_utc(usage_over_time.get("resets_at") or usage_over_time.get("period_ends_at")),
            "provider_limit": provider_limit,
        }

    used_percent = maybe_int(limit_payload.get("used_percent"))
    if used_percent is None:
        return None

    period_duration_mins = maybe_int(
        limit_payload.get("period_duration_mins") or limit_payload.get("window_duration_mins")
    )
    period_started_at = parse_utc(limit_payload.get("period_started_at"))
    period_ends_at = parse_utc(limit_payload.get("period_ends_at") or limit_payload.get("resets_at"))
    resets_at = parse_utc(limit_payload.get("resets_at") or limit_payload.get("period_ends_at"))

    if provider == "codex" or "window_key" in limit_payload or "limit_id" in limit_payload:
        window_key = maybe_str(limit_payload.get("window_key"))
        limit_name = codex_window_name(period_duration_mins, window_key)
        limit_label = codex_window_label(period_duration_mins, window_key)
    elif provider == "claude-code" or "limit_type" in limit_payload:
        limit_type = maybe_str(limit_payload.get("limit_type"))
        limit_name = claude_limit_name(limit_type)
        limit_label = claude_limit_label(limit_type)
    else:
        limit_label = maybe_str(limit_payload.get("limit_label")) or "usage"
        limit_name = copilot_limit_name(limit_label)

    return {
        "limit_class": "usage_over_time" if period_started_at or period_ends_at or period_duration_mins else "usage_snapshot",
        "limit_name": limit_name,
        "limit_label": limit_label,
        "used_percent": used_percent,
        "period_started_at": period_started_at,
        "period_ends_at": period_ends_at,
        "period_duration_mins": period_duration_mins,
        "resets_at": resets_at,
        "provider_limit": {},
    }

def iter_tape_records(tape_root: Path) -> list[dict[str, Any]]:
    segment_paths = sorted((tape_root / "segments").glob("*.jsonl"))
    for segment_path in segment_paths:
        with segment_path.open(encoding="utf-8") as handle:
            for line_no, line in enumerate(handle, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    payload = json.loads(line)
                except json.JSONDecodeError as exc:
                    raise RuntimeError(f"invalid json in {segment_path}:{line_no}: {exc}") from exc
                if isinstance(payload, dict):
                    yield payload

def build_usage_dataset(tape_root: Path) -> dict[str, Any]:
    snapshot_rows: list[dict[str, Any]] = []
    account_rows: list[dict[str, Any]] = []
    limit_rows: list[dict[str, Any]] = []
    aggregate_rows: list[dict[str, Any]] = []

    for record in iter_tape_records(tape_root):
        recorded_at = parse_utc(record.get("recorded_at"))
        if recorded_at is None:
            continue
        recorded_day = recorded_at.date().isoformat()
        providers = record.get("providers")
        if not isinstance(providers, list):
            providers = []

        snapshot_rows.append(
            {
                "recorded_at": recorded_at,
                "recorded_day": recorded_day,
                "provider_count": maybe_int(record.get("provider_count")) or 0,
                "account_count": maybe_int(record.get("account_count")) or 0,
                "ok_count": maybe_int(record.get("ok_count")) or 0,
                "error_count": maybe_int(record.get("error_count")) or 0,
            }
        )

        usage_total = record.get("accounts_usage_total")
        if isinstance(usage_total, dict):
            aggregate = usage_total.get("aggregate")
            if isinstance(aggregate, dict):
                aggregate_rows.append(
                    {
                        "recorded_at": recorded_at,
                        "recorded_day": recorded_day,
                        "label": maybe_str(usage_total.get("label")) or "Selected accounts aggregate",
                        "used_percent": maybe_int(aggregate.get("used_percent")),
                        "elapsed_percent": maybe_int(aggregate.get("elapsed_percent")),
                        "ratio": aggregate.get("ratio"),
                        "included_count": maybe_int(usage_total.get("included_count")),
                    }
                )

        for provider_payload in providers:
            if not isinstance(provider_payload, dict):
                continue
            provider = maybe_str(provider_payload.get("provider")) or "unknown"
            accounts = provider_payload.get("accounts")
            if not isinstance(accounts, list):
                continue

            for account in accounts:
                if not isinstance(account, dict):
                    continue
                account_id = maybe_str(account.get("id")) or "unknown"
                account_label = maybe_str(account.get("label")) or account_id
                account_key = f"{provider}:{account_id}"
                account_rows.append(
                    {
                        "recorded_at": recorded_at,
                        "recorded_day": recorded_day,
                        "provider": provider,
                        "account_id": account_id,
                        "account_label": account_label,
                        "account_key": account_key,
                        "check_status": maybe_str(account.get("check_status")) or "unknown",
                        "usage_status": maybe_str(account.get("usage_status")),
                        "usage_summary": maybe_str(account.get("usage_summary")),
                    }
                )

                usage = account.get("usage")
                if not isinstance(usage, dict):
                    continue
                raw_limits = usage.get("limits")
                if not isinstance(raw_limits, list):
                    continue

                for raw_limit in raw_limits:
                    if not isinstance(raw_limit, dict):
                        continue
                    normalized = normalize_limit_payload(raw_limit, provider)
                    if normalized is None:
                        continue
                    elapsed_percent = compute_elapsed_percent(
                        recorded_at,
                        normalized["period_started_at"],
                        normalized["period_ends_at"],
                    )
                    used_percent = normalized["used_percent"]
                    usage_gap = None
                    if isinstance(used_percent, int) and isinstance(elapsed_percent, int):
                        usage_gap = used_percent - elapsed_percent
                    series_key = f"{account_key}:{normalized['limit_name']}"
                    series_label = f"{provider_title(provider)} / {account_label} / {normalized['limit_label']}"
                    period_signature = (
                        iso_utc(normalized["period_started_at"]),
                        iso_utc(normalized["period_ends_at"]),
                        iso_utc(normalized["resets_at"]),
                        normalized["period_duration_mins"],
                    )
                    limit_rows.append(
                        {
                            "recorded_at": recorded_at,
                            "recorded_day": recorded_day,
                            "provider": provider,
                            "account_id": account_id,
                            "account_label": account_label,
                            "account_key": account_key,
                            "limit_name": normalized["limit_name"],
                            "limit_label": normalized["limit_label"],
                            "limit_class": normalized["limit_class"],
                            "series_key": series_key,
                            "series_label": series_label,
                            "used_percent": used_percent,
                            "elapsed_percent": elapsed_percent,
                            "usage_gap": usage_gap,
                            "period_started_at": normalized["period_started_at"],
                            "period_ends_at": normalized["period_ends_at"],
                            "period_duration_mins": normalized["period_duration_mins"],
                            "resets_at": normalized["resets_at"],
                            "period_signature": period_signature,
                        }
                    )

    snapshot_rows.sort(key=lambda row: row["recorded_at"])
    account_rows.sort(key=lambda row: (row["recorded_at"], row["account_key"]))
    limit_rows.sort(key=lambda row: (row["recorded_at"], row["series_key"]))
    aggregate_rows.sort(key=lambda row: row["recorded_at"])
    return {
        "snapshot_rows": snapshot_rows,
        "account_rows": account_rows,
        "limit_rows": limit_rows,
        "aggregate_rows": aggregate_rows,
    }

def build_speed_rows(limit_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
    for row in limit_rows:
        if isinstance(row.get("used_percent"), int):
            grouped[row["series_key"]].append(row)

    speed_rows: list[dict[str, Any]] = []
    for series_key, rows in grouped.items():
        rows = sorted(rows, key=lambda row: row["recorded_at"])
        previous: dict[str, Any] | None = None
        for row in rows:
            if previous is not None and row["period_signature"] == previous["period_signature"]:
                delta_minutes = (row["recorded_at"] - previous["recorded_at"]).total_seconds() / 60.0
                delta_used = row["used_percent"] - previous["used_percent"]
                if delta_minutes > 0 and delta_used >= 0:
                    speed_rows.append(
                        {
                            "series_key": series_key,
                            "series_label": row["series_label"],
                            "provider": row["provider"],
                            "account_key": row["account_key"],
                            "account_label": row["account_label"],
                            "limit_name": row["limit_name"],
                            "started_at": previous["recorded_at"],
                            "ended_at": row["recorded_at"],
                            "delta_minutes": delta_minutes,
                            "delta_used": delta_used,
                            "percent_per_minute": delta_used / delta_minutes,
                            "percent_per_hour": (delta_used / delta_minutes) * 60.0,
                        }
                    )
            previous = row

    speed_rows.sort(key=lambda row: row["ended_at"])
    return speed_rows

In [ ]:
def stable_index(name: str, size: int) -> int:
    digest = hashlib.sha1(name.encode("utf-8")).digest()
    return int.from_bytes(digest[:4], "big") % size

def hex_to_rgb(color: str) -> tuple[int, int, int]:
    color = color.lstrip("#")
    return tuple(int(color[index : index + 2], 16) for index in (0, 2, 4))

def rgb_to_hex(rgb: tuple[int, int, int]) -> str:
    return "#" + "".join(f"{max(0, min(255, channel)):02x}" for channel in rgb)

def blend(color_a: str, color_b: str, weight: float) -> str:
    a = hex_to_rgb(color_a)
    b = hex_to_rgb(color_b)
    mixed = tuple(round((1 - weight) * left + weight * right) for left, right in zip(a, b))
    return rgb_to_hex(mixed)

def color_for(name: str, provider: str | None = None) -> str:
    base = PROVIDER_COLORS.get(provider or "", PALETTE[stable_index(name, len(PALETTE))])
    accent = PALETTE[stable_index(name + "-accent", len(PALETTE))]
    return blend(base, accent, 0.18)

def format_percent(value: object, signed: bool = False) -> str:
    if value is None:
        return "n/a"
    if signed:
        return f"{int(value):+d}%"
    return f"{int(value)}%"

def metric_cards_html(cards: list[dict[str, str]]) -> str:
    blocks = []
    for card in cards:
        blocks.append(
            f"""
            <div class='usage-card'>
              <div class='usage-kicker'>{html.escape(card['label'])}</div>
              <div class='usage-value'>{html.escape(card['value'])}</div>
              <div class='usage-note'>{html.escape(card['note'])}</div>
            </div>
            """
        )
    return "<div class='usage-wrap usage-grid usage-cards'>" + "".join(blocks) + "</div>"

def render_table(title: str, subtitle: str, columns: list[str], rows: list[list[str]]) -> str:
    header_html = "".join(f"<th>{html.escape(column)}</th>" for column in columns)
    body_html = "".join(
        "<tr>" + "".join(f"<td>{html.escape(value)}</td>" for value in row) + "</tr>"
        for row in rows
    )
    return f"""
    <div class='usage-wrap usage-panel'>
      <h3>{html.escape(title)}</h3>
      <p>{html.escape(subtitle)}</p>
      <table class='usage-table'>
        <thead><tr>{header_html}</tr></thead>
        <tbody>{body_html}</tbody>
      </table>
    </div>
    """

def axis_time_label(value: dt.datetime, span: dt.timedelta) -> str:
    if span.days >= 10:
        return value.strftime("%b %d")
    if span.days >= 1:
        return value.strftime("%b %d %H:%M")
    return value.strftime("%H:%M")

def render_line_chart(
    series_list: list[dict[str, Any]],
    *,
    title: str,
    subtitle: str,
    width: int = 1100,
    height: int = 320,
    y_domain: tuple[float, float] | None = None,
    y_ticks: int = 5,
    baseline_values: tuple[float, ...] = (),
    unit: str = "%",
) -> str:
    filtered_series = []
    x_values: list[dt.datetime] = []
    y_values: list[float] = []
    for series in series_list:
        points = [(point[0], point[1], point[2]) for point in series["points"] if point[1] is not None]
        if not points:
            continue
        filtered_series.append({**series, "points": points})
        x_values.extend(point[0] for point in points)
        y_values.extend(point[1] for point in points)

    if not filtered_series:
        return f"""
        <div class='usage-wrap usage-panel'>
          <h3>{html.escape(title)}</h3>
          <p>{html.escape(subtitle)}</p>
          <div class='usage-note'>No points available for this view.</div>
        </div>
        """

    x_min = min(x_values)
    x_max = max(x_values)
    if x_min == x_max:
        x_min = x_min - dt.timedelta(minutes=30)
        x_max = x_max + dt.timedelta(minutes=30)
    span = x_max - x_min

    if y_domain is None:
        y_min = min(y_values)
        y_max = max(y_values)
        if y_min == y_max:
            pad = 1.0 if y_min == 0 else abs(y_min) * 0.1
        else:
            pad = (y_max - y_min) * 0.08
        y_min -= pad
        y_max += pad
        if y_min > 0:
            y_min = 0.0
        if y_max < 0:
            y_max = 0.0
    else:
        y_min, y_max = y_domain
    if y_max <= y_min:
        y_max = y_min + 1.0

    margin_left = 56
    margin_right = 22
    margin_top = 18
    margin_bottom = 42
    inner_width = width - margin_left - margin_right
    inner_height = height - margin_top - margin_bottom

    def x_scale(value: dt.datetime) -> float:
        return margin_left + ((value - x_min).total_seconds() / span.total_seconds()) * inner_width

    def y_scale(value: float) -> float:
        return margin_top + (1.0 - ((value - y_min) / (y_max - y_min))) * inner_height

    svg_parts = [
        f"<rect x='0' y='0' width='{width}' height='{height}' rx='18' fill='{SURFACE}' />"
    ]

    for tick_index in range(y_ticks + 1):
        value = y_min + ((y_max - y_min) * tick_index / y_ticks)
        y_pos = y_scale(value)
        label = f"{value:.0f}{unit}" if unit else f"{value:.0f}"
        svg_parts.append(
            f"<line x1='{margin_left}' y1='{y_pos:.2f}' x2='{width - margin_right}' y2='{y_pos:.2f}' stroke='{GRID}' stroke-width='1' />"
        )
        svg_parts.append(
            f"<text x='{margin_left - 10}' y='{y_pos + 4:.2f}' fill='{MUTED}' font-size='11' text-anchor='end'>{html.escape(label)}</text>"
        )

    for baseline in baseline_values:
        if baseline < y_min or baseline > y_max:
            continue
        y_pos = y_scale(baseline)
        svg_parts.append(
            f"<line x1='{margin_left}' y1='{y_pos:.2f}' x2='{width - margin_right}' y2='{y_pos:.2f}' stroke='{TEXT}' stroke-width='1.2' stroke-dasharray='5 5' opacity='0.45' />"
        )

    tick_count = 6
    for tick_index in range(tick_count):
        ratio = tick_index / max(1, tick_count - 1)
        tick_time = x_min + dt.timedelta(seconds=span.total_seconds() * ratio)
        x_pos = x_scale(tick_time)
        svg_parts.append(
            f"<line x1='{x_pos:.2f}' y1='{margin_top + inner_height}' x2='{x_pos:.2f}' y2='{margin_top + inner_height + 5}' stroke='{GRID}' stroke-width='1' />"
        )
        svg_parts.append(
            f"<text x='{x_pos:.2f}' y='{height - 12}' fill='{MUTED}' font-size='11' text-anchor='middle'>{html.escape(axis_time_label(tick_time, span))}</text>"
        )

    legend_items = []
    for series in filtered_series:
        color = series.get("color") or color_for(series["name"], series.get("provider"))
        points = sorted(series["points"], key=lambda point: point[0])
        polyline_points = " ".join(
            f"{x_scale(point[0]):.2f},{y_scale(point[1]):.2f}"
            for point in points
        )
        stroke_width = series.get("stroke_width", 2.6)
        dasharray = series.get("dasharray")
        dash_attr = f" stroke-dasharray='{dasharray}'" if dasharray else ""
        svg_parts.append(
            f"<polyline fill='none' stroke='{color}' stroke-width='{stroke_width}' stroke-linejoin='round' stroke-linecap='round'{dash_attr} points='{polyline_points}' />"
        )
        if len(points) <= 80:
            for point in points:
                tooltip = html.escape(point[2])
                svg_parts.append(
                    f"<circle cx='{x_scale(point[0]):.2f}' cy='{y_scale(point[1]):.2f}' r='2.7' fill='{color}'><title>{tooltip}</title></circle>"
                )
        legend_items.append(
            f"<span class='usage-legend-item'><span class='usage-dot' style='background:{color}'></span>{html.escape(series['name'])}</span>"
        )

    svg = f"<svg viewBox='0 0 {width} {height}' width='100%'>{''.join(svg_parts)}</svg>"
    legend_html = "<div class='usage-legend'>" + "".join(legend_items) + "</div>"
    return f"""
    <div class='usage-wrap usage-panel'>
      <h3>{html.escape(title)}</h3>
      <p>{html.escape(subtitle)}</p>
      {svg}
      {legend_html}
    </div>
    """

def render_bar_chart(
    items: list[dict[str, Any]],
    *,
    title: str,
    subtitle: str,
    width: int = 1100,
    height: int = 260,
    unit: str = "",
) -> str:
    if not items:
        return f"""
        <div class='usage-wrap usage-panel'>
          <h3>{html.escape(title)}</h3>
          <p>{html.escape(subtitle)}</p>
          <div class='usage-note'>No bars available for this view.</div>
        </div>
        """

    values = [item["value"] for item in items]
    max_value = max(values) or 1
    margin_left = 52
    margin_right = 20
    margin_top = 18
    margin_bottom = 42
    inner_width = width - margin_left - margin_right
    inner_height = height - margin_top - margin_bottom
    bar_width = inner_width / max(1, len(items))

    svg_parts = [
        f"<rect x='0' y='0' width='{width}' height='{height}' rx='18' fill='{SURFACE}' />"
    ]
    for tick_index in range(5):
        value = max_value * tick_index / 4
        y_pos = margin_top + inner_height - ((value / max_value) * inner_height)
        label = f"{value:.0f}{unit}" if unit else f"{value:.0f}"
        svg_parts.append(
            f"<line x1='{margin_left}' y1='{y_pos:.2f}' x2='{width - margin_right}' y2='{y_pos:.2f}' stroke='{GRID}' stroke-width='1' />"
        )
        svg_parts.append(
            f"<text x='{margin_left - 10}' y='{y_pos + 4:.2f}' fill='{MUTED}' font-size='11' text-anchor='end'>{html.escape(label)}</text>"
        )

    label_step = max(1, math.ceil(len(items) / 12))
    for index, item in enumerate(items):
        value = item["value"]
        x_pos = margin_left + index * bar_width + 1.5
        bar_height = (value / max_value) * inner_height if max_value else 0
        y_pos = margin_top + inner_height - bar_height
        color = item.get("color") or PALETTE[index % len(PALETTE)]
        tooltip = html.escape(f"{item['label']}: {value}{unit}")
        svg_parts.append(
            f"<rect x='{x_pos:.2f}' y='{y_pos:.2f}' width='{max(1.0, bar_width - 3.0):.2f}' height='{bar_height:.2f}' rx='4' fill='{color}' opacity='0.9'><title>{tooltip}</title></rect>"
        )
        if index % label_step == 0:
            svg_parts.append(
                f"<text x='{x_pos + (bar_width - 3.0) / 2:.2f}' y='{height - 12}' fill='{MUTED}' font-size='10' text-anchor='middle'>{html.escape(str(item['label']))}</text>"
            )

    svg = f"<svg viewBox='0 0 {width} {height}' width='100%'>{''.join(svg_parts)}</svg>"
    return f"""
    <div class='usage-wrap usage-panel'>
      <h3>{html.escape(title)}</h3>
      <p>{html.escape(subtitle)}</p>
      {svg}
    </div>
    """

def render_sparkline(points: list[tuple[dt.datetime, int]], color: str, width: int = 180, height: int = 48) -> str:
    if not points:
        return ""
    points = sorted(points, key=lambda point: point[0])
    x_min = points[0][0]
    x_max = points[-1][0]
    if x_min == x_max:
        x_min = x_min - dt.timedelta(minutes=30)
        x_max = x_max + dt.timedelta(minutes=30)
    span = x_max - x_min
    margin = 4
    inner_width = width - margin * 2
    inner_height = height - margin * 2

    def x_scale(value: dt.datetime) -> float:
        return margin + ((value - x_min).total_seconds() / span.total_seconds()) * inner_width

    def y_scale(value: float) -> float:
        return margin + (1.0 - (value / 100.0)) * inner_height

    polyline = " ".join(f"{x_scale(x):.2f},{y_scale(y):.2f}" for x, y in points)
    return f"""
    <svg viewBox='0 0 {width} {height}' width='100%' height='{height}'>
      <rect x='0' y='0' width='{width}' height='{height}' rx='10' fill='{SURFACE}' />
      <polyline fill='none' stroke='{color}' stroke-width='2.2' stroke-linecap='round' stroke-linejoin='round' points='{polyline}' />
    </svg>
    """

def latest_rows_by_key(rows: list[dict[str, Any]], key_field: str) -> dict[str, dict[str, Any]]:
    latest: dict[str, dict[str, Any]] = {}
    for row in rows:
        latest[row[key_field]] = row
    return latest

def render_series_cards(
    latest_rows: list[dict[str, Any]],
    history_by_series: dict[str, list[dict[str, Any]]],
    speed_by_series: dict[str, dict[str, Any]],
    *,
    limit: int = 8,
) -> str:
    blocks = []
    for row in latest_rows[:limit]:
        provider = row["provider"]
        color = color_for(row["series_key"], provider)
        spark_points = [
            (item["recorded_at"], item["used_percent"])
            for item in history_by_series.get(row["series_key"], [])
            if isinstance(item.get("used_percent"), int)
        ]
        spark = render_sparkline(spark_points, color)
        speed = speed_by_series.get(row["series_key"])
        speed_text = "n/a"
        if speed is not None:
            speed_text = f"{speed['percent_per_hour']:.1f}%/h"
        note = (
            f"{provider_title(provider)} | {pretty_ts(row['recorded_at'])} | "
            f"gap {format_percent(row['usage_gap'], signed=True)} | burn {speed_text}"
        )
        blocks.append(
            f"""
            <div class='usage-card'>
              <div class='usage-kicker'>{html.escape(row['series_label'])}</div>
              <div class='usage-value'>{html.escape(format_percent(row['used_percent']))}</div>
              <div class='usage-note'>{html.escape(note)}</div>
              <div style='margin-top:10px'>{spark}</div>
            </div>
            """
        )
    return "<div class='usage-wrap usage-grid usage-cards'>" + "".join(blocks) + "</div>"

In [ ]:
DATA = build_usage_dataset(TAPE_ROOT)
SNAPSHOT_ROWS = DATA["snapshot_rows"]
ACCOUNT_ROWS = DATA["account_rows"]
LIMIT_ROWS = DATA["limit_rows"]
AGGREGATE_ROWS = DATA["aggregate_rows"]
SPEED_ROWS = build_speed_rows(LIMIT_ROWS)

first_snapshot = SNAPSHOT_ROWS[0]["recorded_at"] if SNAPSHOT_ROWS else None
last_snapshot = SNAPSHOT_ROWS[-1]["recorded_at"] if SNAPSHOT_ROWS else None
unique_providers = sorted({row["provider"] for row in ACCOUNT_ROWS})
unique_accounts = sorted({row["account_key"] for row in ACCOUNT_ROWS})
unique_series = sorted({row["series_key"] for row in LIMIT_ROWS})
days_covered = 0
if first_snapshot and last_snapshot:
    days_covered = (last_snapshot.date() - first_snapshot.date()).days + 1

cards = [
    {
        "label": "Snapshots",
        "value": f"{len(SNAPSHOT_ROWS):,}",
        "note": f"{days_covered} days of capture history",
    },
    {
        "label": "Accounts Seen",
        "value": f"{len(unique_accounts)}",
        "note": ", ".join(provider_title(provider) for provider in unique_providers),
    },
    {
        "label": "Limit Series",
        "value": f"{len(unique_series)}",
        "note": "normalized across schema changes",
    },
    {
        "label": "Latest Snapshot",
        "value": pretty_ts(last_snapshot),
        "note": str((last_snapshot - first_snapshot).days) + " days after first capture" if first_snapshot and last_snapshot else "n/a",
    },
]
display(HTML(metric_cards_html(cards)))

daily_snapshot_counts: Counter[str] = Counter(row["recorded_day"] for row in SNAPSHOT_ROWS)
daily_snapshot_items = [
    {"label": day[5:], "value": count, "color": "#b45309"}
    for day, count in sorted(daily_snapshot_counts.items())[-45:]
]
display(
    HTML(
        render_bar_chart(
            daily_snapshot_items,
            title="Snapshot cadence",
            subtitle="How often the account tape was refreshed per day over the last 45 captured days.",
            unit="",
        )
    )
)

if AGGREGATE_ROWS:
    aggregate_label = AGGREGATE_ROWS[-1]["label"]
    recent_aggregate = AGGREGATE_ROWS[-1200:]
    aggregate_series = [
        {
            "name": f"{aggregate_label} used",
            "provider": None,
            "color": "#111827",
            "points": [
                (
                    row["recorded_at"],
                    row["used_percent"],
                    f"{pretty_ts(row['recorded_at'])} | used {format_percent(row['used_percent'])} | elapsed {format_percent(row['elapsed_percent'])}",
                )
                for row in recent_aggregate
                if row["used_percent"] is not None
            ],
        },
        {
            "name": f"{aggregate_label} elapsed",
            "provider": None,
            "color": "#d97706",
            "dasharray": "6 5",
            "points": [
                (
                    row["recorded_at"],
                    row["elapsed_percent"],
                    f"{pretty_ts(row['recorded_at'])} | elapsed {format_percent(row['elapsed_percent'])}",
                )
                for row in recent_aggregate
                if row["elapsed_percent"] is not None
            ],
        },
    ]
    display(
        HTML(
            render_line_chart(
                aggregate_series,
                title="Selected-accounts aggregate",
                subtitle="This appears only after `agent.accounts.usage_total` was added to the refresh payloads.",
                y_domain=(0, 100),
                baseline_values=(100,),
            )
        )
    )

latest_accounts = latest_rows_by_key(ACCOUNT_ROWS, "account_key")
account_table_rows = []
for account_key, row in sorted(latest_accounts.items(), key=lambda item: item[1]["provider"] + item[1]["account_label"]):
    account_table_rows.append(
        [
            provider_title(row["provider"]),
            row["account_label"],
            row["check_status"],
            row["usage_status"] or "-",
            pretty_ts(row["recorded_at"]),
        ]
    )
display(
    HTML(
        render_table(
            "Latest account snapshot",
            "The most recent known status row for each account instance seen in the tape.",
            ["Provider", "Account", "Check", "Usage", "Recorded"],
            account_table_rows,
        )
    )
)

## Interactive Explorer

Use the filters below to focus on specific providers, accounts, limits, and lookback windows. The explorer keeps the charts dependency-light by rendering SVG directly inside the notebook.

In [ ]:
def account_options_for_provider(provider_value: str) -> list[tuple[str, str]]:
    matching_rows = [
        row for row in ACCOUNT_ROWS
        if provider_value == "all" or row["provider"] == provider_value
    ]
    labels = {}
    for row in matching_rows:
        labels[row["account_key"]] = f"{provider_title(row['provider'])} / {row['account_label']}"
    return sorted((label, key) for key, label in labels.items())

def limit_options_for_selection(provider_value: str, account_values: tuple[str, ...]) -> list[tuple[str, str]]:
    matching_rows = []
    for row in LIMIT_ROWS:
        if provider_value != "all" and row["provider"] != provider_value:
            continue
        if account_values and row["account_key"] not in account_values:
            continue
        matching_rows.append(row)
    labels = {}
    for row in matching_rows:
        labels[row["limit_name"]] = row["limit_label"]
    return sorted((label, key) for key, label in labels.items())

provider_widget = widgets.Dropdown(
    description="Provider",
    options=[("All providers", "all")] + [
        (provider_title(provider), provider) for provider in sorted({row["provider"] for row in ACCOUNT_ROWS})
    ],
    value="all",
    layout=widgets.Layout(width="260px"),
)
account_widget = widgets.SelectMultiple(
    description="Accounts",
    options=account_options_for_provider("all"),
    rows=7,
    layout=widgets.Layout(width="360px"),
)
limit_widget = widgets.SelectMultiple(
    description="Limits",
    options=limit_options_for_selection("all", ()),
    rows=5,
    layout=widgets.Layout(width="240px"),
)
lookback_widget = widgets.IntSlider(
    description="Lookback days",
    min=3,
    max=60,
    step=1,
    value=21,
    continuous_update=False,
    layout=widgets.Layout(width="340px"),
)
max_series_widget = widgets.IntSlider(
    description="Max series",
    min=1,
    max=10,
    step=1,
    value=6,
    continuous_update=False,
    layout=widgets.Layout(width="320px"),
)
output = widgets.Output()

def update_filter_options(*_args: Any) -> None:
    account_options = account_options_for_provider(provider_widget.value)
    valid_accounts = {value for _, value in account_options}
    account_widget.options = account_options
    account_widget.value = tuple(value for value in account_widget.value if value in valid_accounts)

    limit_options = limit_options_for_selection(provider_widget.value, account_widget.value)
    valid_limits = {value for _, value in limit_options}
    limit_widget.options = limit_options
    limit_widget.value = tuple(value for value in limit_widget.value if value in valid_limits)

def selected_limit_rows() -> list[dict[str, Any]]:
    if not LIMIT_ROWS:
        return []
    anchor_time = LIMIT_ROWS[-1]["recorded_at"]
    start_time = anchor_time - dt.timedelta(days=lookback_widget.value)
    rows = []
    for row in LIMIT_ROWS:
        if row["recorded_at"] < start_time:
            continue
        if provider_widget.value != "all" and row["provider"] != provider_widget.value:
            continue
        if account_widget.value and row["account_key"] not in account_widget.value:
            continue
        if limit_widget.value and row["limit_name"] not in limit_widget.value:
            continue
        rows.append(row)
    return rows

def selected_account_rows() -> list[dict[str, Any]]:
    if not ACCOUNT_ROWS:
        return []
    anchor_time = ACCOUNT_ROWS[-1]["recorded_at"]
    start_time = anchor_time - dt.timedelta(days=lookback_widget.value)
    rows = []
    for row in ACCOUNT_ROWS:
        if row["recorded_at"] < start_time:
            continue
        if provider_widget.value != "all" and row["provider"] != provider_widget.value:
            continue
        if account_widget.value and row["account_key"] not in account_widget.value:
            continue
        rows.append(row)
    return rows

def render_dashboard(*_args: Any) -> None:
    rows = selected_limit_rows()
    account_rows = selected_account_rows()
    with output:
        clear_output(wait=True)
        if not rows:
            display(HTML("<div class='usage-wrap usage-panel'><h3>No matching rows</h3><p>Change the filters to pull more series into view.</p></div>"))
            return

        grouped: dict[str, list[dict[str, Any]]] = defaultdict(list)
        for row in rows:
            grouped[row["series_key"]].append(row)
        latest = [values[-1] for _, values in grouped.items()]
        latest.sort(
            key=lambda row: (
                row["recorded_at"],
                row["used_percent"] if isinstance(row["used_percent"], int) else -1,
                row["series_label"],
            ),
            reverse=True,
        )
        selected_latest = latest[: max_series_widget.value]
        selected_keys = {row["series_key"] for row in selected_latest}
        grouped = {key: grouped[key] for key in selected_keys}

        line_series = []
        gap_series = []
        for row in selected_latest:
            series_rows = grouped[row["series_key"]]
            color = color_for(row["series_key"], row["provider"])
            line_series.append(
                {
                    "name": row["series_label"],
                    "provider": row["provider"],
                    "color": color,
                    "points": [
                        (
                            item["recorded_at"],
                            item["used_percent"],
                            f"{item['series_label']} | {pretty_ts(item['recorded_at'])} | used {format_percent(item['used_percent'])} | elapsed {format_percent(item['elapsed_percent'])}",
                        )
                        for item in series_rows
                        if item["used_percent"] is not None
                    ],
                }
            )
            gap_points = [
                (
                    item["recorded_at"],
                    item["usage_gap"],
                    f"{item['series_label']} | {pretty_ts(item['recorded_at'])} | gap {format_percent(item['usage_gap'], signed=True)}",
                )
                for item in series_rows
                if item["usage_gap"] is not None
            ]
            if gap_points:
                gap_series.append(
                    {
                        "name": row["series_label"],
                        "provider": row["provider"],
                        "color": color,
                        "points": gap_points,
                    }
                )

        error_counts = Counter(
            row["recorded_day"]
            for row in account_rows
            if row["check_status"] != "ok"
        )
        error_items = [
            {"label": day[5:], "value": error_counts[day], "color": "#dc2626"}
            for day in sorted(error_counts)
        ]
        if not error_items:
            all_days = sorted({row["recorded_day"] for row in account_rows})
            error_items = [
                {"label": day[5:], "value": 0, "color": "#dc2626"}
                for day in all_days[-30:]
            ]

        speed_rows = [
            row for row in SPEED_ROWS
            if row["series_key"] in selected_keys
        ]
        speed_by_series = latest_rows_by_key(speed_rows, "series_key")
        history_by_series = {key: values for key, values in grouped.items()}

        cards = [
            {
                "label": "Visible series",
                "value": str(len(selected_keys)),
                "note": f"from {len(rows):,} filtered limit rows",
            },
            {
                "label": "Accounts in range",
                "value": str(len({row['account_key'] for row in account_rows})),
                "note": f"{lookback_widget.value}-day lookback",
            },
            {
                "label": "Errors in range",
                "value": str(sum(item['value'] for item in error_items)),
                "note": "daily account check failures",
            },
            {
                "label": "Latest plotted point",
                "value": pretty_ts(max(row['recorded_at'] for row in rows)),
                "note": "latest matching limit observation",
            },
        ]
        display(HTML(metric_cards_html(cards)))
        display(
            HTML(
                render_line_chart(
                    line_series,
                    title="Usage trajectories",
                    subtitle="Used percent over time for the most recent matching series.",
                    y_domain=(0, 100),
                    baseline_values=(100,),
                )
            )
        )
        if gap_series:
            display(
                HTML(
                    render_line_chart(
                        gap_series,
                        title="Lead or lag versus cycle progress",
                        subtitle="Positive values mean usage is running ahead of elapsed cycle time; negative values mean it is behind.",
                        y_domain=(-100, 100),
                        baseline_values=(0,),
                    )
                )
            )
        display(
            HTML(
                render_bar_chart(
                    error_items[-30:],
                    title="Daily error counts",
                    subtitle="Account check failures across the current filter range.",
                )
            )
        )
        display(HTML(render_series_cards(selected_latest, history_by_series, speed_by_series, limit=max_series_widget.value)))

        top_speed_rows = sorted(speed_rows, key=lambda row: row["percent_per_hour"], reverse=True)[:10]
        if top_speed_rows:
            display(
                HTML(
                    render_table(
                        "Fastest burn-rate intervals",
                        "Only consecutive points inside the same reset window are compared.",
                        ["Series", "From", "To", "Delta", "Minutes", "%/hour"],
                        [
                            [
                                row["series_label"],
                                pretty_ts(row["started_at"]),
                                pretty_ts(row["ended_at"]),
                                format_percent(row["delta_used"]),
                                f"{row['delta_minutes']:.1f}",
                                f"{row['percent_per_hour']:.2f}",
                            ]
                            for row in top_speed_rows
                        ],
                    )
                )
            )

for widget in (provider_widget, account_widget, limit_widget, lookback_widget, max_series_widget):
    widget.observe(update_filter_options if widget is provider_widget or widget is account_widget else render_dashboard, names="value")
provider_widget.observe(render_dashboard, names="value")
account_widget.observe(render_dashboard, names="value")

update_filter_options()
controls = widgets.VBox(
    [
        widgets.HBox([provider_widget, lookback_widget, max_series_widget]),
        widgets.HBox([account_widget, limit_widget]),
    ]
)
display(controls)
render_dashboard()
display(output)